# 21 Teacher Feedback Normalised Evaluation

Implements the normalised regional evaluation developed in response to teacher feedback.

**Provenance.** This is an English-facing execution copy of the original project notebook. The computational cells are preserved so that the analysis remains reproducible. Chinese narrative cells have been replaced by concise English stage notes; the original notebook remains unchanged in the source project.

## Analysis stage 1

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
from pathlib import Path 
import importlib .util 
import warnings 

from IPython import get_ipython 

# Implementation note translated in the organised English notebook.
# Implementation note translated in the organised English notebook.
# Implementation note translated in the organised English notebook.
_ipython =get_ipython ()
if _ipython is not None :
    _ipython .run_line_magic ("matplotlib","inline")

import matplotlib 
import matplotlib .pyplot as plt 
import numpy as np 
import pandas as pd 
from IPython .display import display ,Markdown 

warnings .filterwarnings ("ignore",category =FutureWarning )

PROJECT_ROOT =next (
(p for p in [Path .cwd (),*Path .cwd ().parents ]if (p /"RQ2").exists ()and (p /"data"/"processed").exists ()),
None ,
)
if PROJECT_ROOT is None :
    raise FileNotFoundError ("\u627e\u4e0d\u5230\u9879\u76ee\u6839\u76ee\u5f55。\u8bf7\u4ece data3001-project \u6216\u5176\u5b50\u76ee\u5f55\u8fd0\u884c。")

RQ2 =PROJECT_ROOT /"RQ2"
DATA_OUT =RQ2 /"data"/"teacher_feedback_extension"
FIG_OUT =RQ2 /"figures"/"teacher_feedback_extension"
DATA_OUT .mkdir (parents =True ,exist_ok =True )
FIG_OUT .mkdir (parents =True ,exist_ok =True )

REGIMES =["Shelf","Kuroshio Core","Transition","Other Offshore"]
MODELS =["Persistence","Mean Flow","VAR(1)","Ridge","KNN","FNN"]
MODEL_COLORS ={
"Persistence":"#636363","Mean Flow":"#9e9e9e","VAR(1)":"#e67e00",
"Ridge":"#2b65d9","KNN":"#159a6b","FNN":"#d81b60",
}
SEED =3001 
N_BOOTSTRAP =2000 
H =72 

plt .style .use ("seaborn-v0_8-whitegrid")
# Implementation note translated in the organised English notebook.
# Implementation note translated in the organised English notebook.
plt .rcParams ["font.sans-serif"]=["DejaVu Sans","PingFang SC","Arial Unicode MS"]
plt .rcParams ["axes.unicode_minus"]=False 

print ("\u9879\u76ee\u76ee\u5f55：",PROJECT_ROOT )
print ("\u7ed3\u679c\u76ee\u5f55：",DATA_OUT )

## Analysis stage 2

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
global_errors =pd .read_csv (
RQ2 /"data"/"regional_lead_time"/"six_model_regional_lead_errors.csv"
)
regional_errors =pd .read_csv (
RQ2 /"data"/"kuroshio_all_models"/"regional_test_lead_errors.csv"
)
regime_labels =pd .read_csv (RQ2 /"data"/"forecast_case_regimes.csv")

checks =pd .Series ({
"global models = 6":global_errors ["model"].nunique ()==6 ,
"global test cases = 757":global_errors ["forecast_case_id"].nunique ()==757 ,
"global lead times = 72":global_errors ["lead_time_hours"].nunique ()==72 ,
"regional learned models = 4":regional_errors ["model"].nunique ()==4 ,
"Kuroshio regional test cases = 139":regional_errors ["forecast_case_id"].nunique ()==139 ,
},name ="\u901a\u8fc7")
display (checks .to_frame ())
assert checks .all ()

## Analysis stage 3

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
def load_module (name ,path ):
    spec =importlib .util .spec_from_file_location (name ,path )
    module =importlib .util .module_from_spec (spec )
    original_use =matplotlib .use 
    try :
    # Implementation note translated in the organised English notebook.
    # Implementation note translated in the organised English notebook.
        matplotlib .use =lambda *args ,**kwargs :None 
        spec .loader .exec_module (module )
    finally :
        matplotlib .use =original_use 
    return module 


ext =load_module ("rq2_extension_18_for_teacher",RQ2 /"18_regime_conditioned_forecasting.py")
data ,targets ,x_original ,_ ,y ,_ =ext .build_xy ()
y_3d =y .reshape (len (data ),H ,2 )

scale_rows =[]
for regime in REGIMES :
    idx =np .flatnonzero (
    data ["partition"].eq ("training").to_numpy ()
    &data ["regime"].eq (regime ).to_numpy ()
    )
    target =y_3d [idx ]
    centre =target .mean (axis =0 )
    vector_scale =np .sqrt (np .mean (np .sum ((target -centre [None ,:,:])**2 ,axis =2 ),axis =0 ))
    displacement_magnitude =np .linalg .norm (target ,axis =2 )
    for j in range (H ):
        scale_rows .append ({
        "regime":regime ,
        "lead_time_hours":j +1 ,
        "n_training_cases":len (idx ),
        "n_training_drifters":data .loc [idx ,"ID"].nunique (),
        "vector_target_scale_km":vector_scale [j ],
        "median_training_displacement_km":np .median (displacement_magnitude [:,j ]),
        "mean_training_displacement_km":np .mean (displacement_magnitude [:,j ]),
        })
training_scale =pd .DataFrame (scale_rows )

display (
training_scale [training_scale ["lead_time_hours"].isin ([24 ,48 ,72 ])]
.round (2 )
)

## Analysis stage 4

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

## Analysis stage 5

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
global_summary = (
    global_errors.groupby(["model", "regime", "lead_time_hours"], as_index=False)
    .agg(
        n_cases=("forecast_case_id", "size"),
        n_drifters=("ID", "nunique"),
        median_error_km=("separation_error_km", "median"),
        rmse_km=("separation_error_km", lambda s: np.sqrt(np.mean(np.square(s)))),
    )
    .merge(
        training_scale[["regime", "lead_time_hours", "vector_target_scale_km", "median_training_displacement_km"]],
        on=["regime", "lead_time_hours"], how="left", validate="many_to_one",
    )
)
global_summary["nrmse"] = global_summary["rmse_km"] / global_summary["vector_target_scale_km"]
global_summary["median_error_over_median_displacement"] = (
    global_summary["median_error_km"] / global_summary["median_training_displacement_km"]
)

# Baseline-relative skill, evaluated on the same cases in each regime and lead time.
persistence_rmse = global_summary[global_summary["model"].eq("Persistence")][
    ["regime", "lead_time_hours", "rmse_km"]
].rename(columns={"rmse_km": "persistence_rmse_km"})
meanflow_rmse = global_summary[global_summary["model"].eq("Mean Flow")][
    ["regime", "lead_time_hours", "rmse_km"]
].rename(columns={"rmse_km": "meanflow_rmse_km"})
global_summary = global_summary.merge(
    persistence_rmse, on=["regime", "lead_time_hours"], validate="many_to_one"
).merge(
    meanflow_rmse, on=["regime", "lead_time_hours"], validate="many_to_one"
)
global_summary["skill_vs_persistence"] = 1 - global_summary["rmse_km"] / global_summary["persistence_rmse_km"]
global_summary["skill_vs_mean_flow"] = 1 - global_summary["rmse_km"] / global_summary["meanflow_rmse_km"]

table_72 = global_summary[global_summary["lead_time_hours"].eq(72)].copy()
display(table_72[[
    "regime", "model", "n_cases", "median_error_km", "rmse_km", "nrmse",
    "skill_vs_persistence", "skill_vs_mean_flow",
]].round(3))

## Analysis stage 6

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

## Analysis stage 7

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
for ax, regime in zip(axes.flat, REGIMES):
    part = global_summary[global_summary["regime"].eq(regime)]
    for model in MODELS:
        m = part[part["model"].eq(model)].sort_values("lead_time_hours")
        ax.plot(m["lead_time_hours"], m["rmse_km"], lw=2,
                color=MODEL_COLORS[model], label=model)
    ax.set_title(regime, fontweight="bold")
    ax.set_ylabel("Absolute RMSE (km)")
    ax.grid(alpha=0.25)
for ax in axes[-1]:
    ax.set_xlabel("Lead time (h)")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=6, frameon=False)
fig.suptitle("Global models evaluated within each regime: absolute error", y=0.98, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(FIG_OUT / "absolute_rmse_by_regime_and_lead.png", dpi=200, bbox_inches="tight")
plt.show()

## Analysis stage 8

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
for ax, regime in zip(axes.flat, REGIMES):
    part = global_summary[global_summary["regime"].eq(regime)]
    for model in MODELS:
        m = part[part["model"].eq(model)].sort_values("lead_time_hours")
        ax.plot(m["lead_time_hours"], m["nrmse"], lw=2,
                color=MODEL_COLORS[model], label=model)
    ax.axhline(1, color="black", ls=":", lw=1)
    ax.set_title(regime, fontweight="bold")
    ax.set_ylabel("NRMSE (training scale = 1)")
    ax.grid(alpha=0.25)
for ax in axes[-1]:
    ax.set_xlabel("Lead time (h)")
fig.legend(handles, labels, loc="upper center", ncol=6, frameon=False)
fig.suptitle("Global models evaluated within each regime: scale-normalised error", y=0.98, fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(FIG_OUT / "normalised_rmse_by_regime_and_lead.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. 72 h forecast skill by regime

In [ ]:
learned = table_72[table_72["model"].isin(["VAR(1)", "Ridge", "KNN", "FNN"])].copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, skill_col, title in zip(
    axes,
    ["skill_vs_persistence", "skill_vs_mean_flow"],
    ["Skill versus velocity persistence", "Skill versus spatial mean flow"],
):
    pivot = learned.pivot(index="regime", columns="model", values=skill_col).reindex(REGIMES)
    pivot = pivot[["VAR(1)", "Ridge", "KNN", "FNN"]]
    pivot.plot(kind="bar", ax=ax, color=[MODEL_COLORS[c] for c in pivot.columns], width=0.8)
    ax.axhline(0, color="black", lw=1)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("72 h forecast skill")
    ax.tick_params(axis="x", rotation=25)
    ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig(FIG_OUT / "forecast_skill_72h_by_regime.png", dpi=200, bbox_inches="tight")
plt.show()

## Analysis stage 9

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
regional_to_global = {
    "Regional FNN": "FNN",
    "Regional Ridge": "Ridge",
    "Regional Weighted KNN": "KNN",
    "Regional VAR(1)": "VAR(1)",
}
global_72 = global_errors[
    global_errors["regime"].eq("Kuroshio Core")
    & global_errors["lead_time_hours"].eq(72)
][["model", "forecast_case_id", "ID", "separation_error_km"]].rename(
    columns={"separation_error_km": "global_error_km"}
)
regional_72 = regional_errors[regional_errors["lead_time_hours"].eq(72)].copy()
regional_72["global_model"] = regional_72["model"].map(regional_to_global)
regional_72 = regional_72.rename(columns={"separation_error_km": "regional_error_km"})

paired = regional_72.merge(
    global_72,
    left_on=["global_model", "forecast_case_id", "ID"],
    right_on=["model", "forecast_case_id", "ID"],
    validate="one_to_one",
    suffixes=("_regional", "_global"),
)

kuro_scale_72 = float(training_scale.query(
    "regime == 'Kuroshio Core' and lead_time_hours == 72"
)["vector_target_scale_km"].iloc[0])
rng = np.random.default_rng(SEED)
bootstrap_rows = []

for regional_model, group in paired.groupby("model_regional"):
    drifters = group["ID"].unique()
    point_global = np.sqrt(np.mean(group["global_error_km"] ** 2))
    point_regional = np.sqrt(np.mean(group["regional_error_km"] ** 2))
    point_improvement = 100 * (1 - point_regional / point_global)
    reps = []
    for _ in range(N_BOOTSTRAP):
        sampled = rng.choice(drifters, size=len(drifters), replace=True)
        sample = pd.concat([group[group["ID"].eq(d)] for d in sampled], ignore_index=True)
        g = np.sqrt(np.mean(sample["global_error_km"] ** 2))
        r = np.sqrt(np.mean(sample["regional_error_km"] ** 2))
        reps.append(100 * (1 - r / g))
    bootstrap_rows.append({
        "model": regional_to_global[regional_model],
        "n_cases": len(group),
        "n_drifters": len(drifters),
        "global_rmse_km": point_global,
        "regional_rmse_km": point_regional,
        "global_nrmse": point_global / kuro_scale_72,
        "regional_nrmse": point_regional / kuro_scale_72,
        "rmse_improvement_pct": point_improvement,
        "improvement_ci_lower_95": np.quantile(reps, 0.025),
        "improvement_ci_upper_95": np.quantile(reps, 0.975),
        "bootstrap_probability_improvement_gt_0": np.mean(np.asarray(reps) > 0),
    })

bootstrap_summary = pd.DataFrame(bootstrap_rows).sort_values("rmse_improvement_pct", ascending=False)
display(bootstrap_summary.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot = bootstrap_summary.reset_index(drop=True)
xpos = np.arange(len(plot))
yerr = np.vstack([
    plot["rmse_improvement_pct"] - plot["improvement_ci_lower_95"],
    plot["improvement_ci_upper_95"] - plot["rmse_improvement_pct"],
])
ax.errorbar(xpos, plot["rmse_improvement_pct"], yerr=yerr, fmt="o", ms=8,
            capsize=5, color="#159FD3", ecolor="#4B5563")
ax.axhline(0, color="black", lw=1, ls="--")
ax.set_xticks(xpos, plot["model"])
ax.set(title="Kuroshio-only training: RMSE improvement with drifter-clustered 95% CIs",
       ylabel="Improvement versus global model (%)")
ax.grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig(FIG_OUT / "kuroshio_regional_improvement_bootstrap_ci.png", dpi=200, bbox_inches="tight")
plt.show()

## Analysis stage 10

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

## Analysis stage 11

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
kmod =load_module ("rq2_kuroshio_models_for_teacher",RQ2 /"19_kuroshio_all_models.py")
kuro_train =np .flatnonzero (
data ["partition"].eq ("training").to_numpy ()
&data ["regime"].eq ("Kuroshio Core").to_numpy ()
)
kuro_test =np .flatnonzero (
data ["partition"].eq ("test").to_numpy ()
&data ["regime"].eq ("Kuroshio Core").to_numpy ()
)
assert len (kuro_train )==646 and len (kuro_test )==139 

# Fixed regional hyperparameters from the existing grouped-CV result.
regional_fnn =kmod .train_fnn (
x_original [kuro_train ],y [kuro_train ],
hidden =(128 ,64 ),l2 =0.001 ,epochs =121 ,seed =kmod .SEED +999 ,
)
regional_prediction =kmod .fnn_predict (regional_fnn ,x_original [kuro_test ]).reshape (-1 ,H ,2 )

# Existing global FNN parameters fitted on all 3,591 training cases.
archive =np .load (PROJECT_ROOT /"output"/"rq1"/"en"/"fnn_model_parameters.npz")
z =(x_original [kuro_test ]-archive ["x_mean"])/archive ["x_scale"]
z =np .maximum (z @archive ["W1"]+archive ["b1"],0 )
z =np .maximum (z @archive ["W2"]+archive ["b2"],0 )
global_prediction =((z @archive ["W3"]+archive ["b3"])*archive ["y_scale"]+archive ["y_mean"]).reshape (-1 ,H ,2 )
true_prediction =y [kuro_test ].reshape (-1 ,H ,2 )

# Confirm deterministic reconstruction against saved 72 h regional FNN errors.
test_cases =data .loc [kuro_test ].reset_index (drop =True )
origin_lat =np .repeat (test_cases ["origin_latitude"].to_numpy ()[:,None ],H ,axis =1 )
origin_lon =np .repeat (test_cases ["origin_longitude"].to_numpy ()[:,None ],H ,axis =1 )

def to_latlon (local ):
    return ext .local_km_to_latlon (origin_lat ,origin_lon ,local [:,:,0 ],local [:,:,1 ])

true_lat ,true_lon =to_latlon (true_prediction )
global_lat ,global_lon =to_latlon (global_prediction )
regional_lat ,regional_lon =to_latlon (regional_prediction )
global_sep =ext .haversine_km (global_lat ,global_lon ,true_lat ,true_lon )
regional_sep =ext .haversine_km (regional_lat ,regional_lon ,true_lat ,true_lon )

saved_regional_fnn_72 =(
regional_errors [
regional_errors ["model"].eq ("Regional FNN")
&regional_errors ["lead_time_hours"].eq (72 )
]
.sort_values ("forecast_case_id")["separation_error_km"].to_numpy ()
)
reconstructed_72 =pd .DataFrame ({
"forecast_case_id":test_cases ["forecast_case_id"],
"error":regional_sep [:,-1 ],
}).sort_values ("forecast_case_id")["error"].to_numpy ()
print ("Regional FNN\u590d\u73b0\u6700\u5927\u5dee\u5f02：",np .max (np .abs (saved_regional_fnn_72 -reconstructed_72 )))
assert np .max (np .abs (saved_regional_fnn_72 -reconstructed_72 ))<1e-8 

In [ ]:
case_improvement = global_sep[:, -1] - regional_sep[:, -1]
order = np.argsort(case_improvement)
selected_positions = [
    order[-1],                              # regional improvement largest
    order[np.argmin(np.abs(case_improvement[order] - np.median(case_improvement)))],
    order[0],                               # regional worsens most
]
selected_titles = ["Largest regional improvement", "Near-median change", "Regional model worsens most"]

fig, axes = plt.subplots(1, 3, figsize=(17, 5.2))
for ax, pos, subtitle in zip(axes, selected_positions, selected_titles):
    ax.plot(true_lon[pos], true_lat[pos], color="#159a6b", lw=2.4, label="Observed")
    ax.plot(global_lon[pos], global_lat[pos], color="#4B5563", lw=2, ls="--", label="Global FNN")
    ax.plot(regional_lon[pos], regional_lat[pos], color="#d81b60", lw=2, ls="-.", label="Kuroshio-only FNN")
    ax.scatter(origin_lon[pos, 0], origin_lat[pos, 0], marker="*", s=120, color="black", zorder=5)
    case_id = int(test_cases.loc[pos, "forecast_case_id"])
    ax.set_title(
        f"{subtitle}\nCase {case_id}: global={global_sep[pos,-1]:.1f} km, regional={regional_sep[pos,-1]:.1f} km",
        fontsize=10, fontweight="bold",
    )
    ax.set_xlabel("Longitude (°E)")
    ax.set_ylabel("Latitude (°N)")
    ax.grid(alpha=0.25)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)
fig.suptitle("What does Kuroshio-only FNN training change in individual trajectories?", y=1.02, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_OUT / "kuroshio_global_vs_regional_fnn_examples.png", dpi=200, bbox_inches="tight")
plt.show()

## Analysis stage 12

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

## Analysis stage 13

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
split_horizon_design = pd.DataFrame({
    "candidate_T_star_h": [24, 36, 48],
    "short_target": ["1-23 h", "1-35 h", "1-47 h"],
    "long_target": ["24-72 h", "36-72 h", "48-72 h"],
    "selection_data": ["training-only grouped 5-fold CV"] * 3,
    "test_usage": ["once, after all choices are frozen"] * 3,
})
display(split_horizon_design)

## Analysis stage 14

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
overall_rows = []
for model in MODELS:
    part = table_72[table_72["model"].eq(model)].set_index("regime").reindex(REGIMES)
    equal_weight = part["nrmse"].mean()
    case_weight = np.average(part["nrmse"], weights=part["n_cases"])
    overall_rows.append({
        "model": model,
        "equal_regime_weight_overall_nrmse": equal_weight,
        "test_case_weighted_overall_nrmse_descriptive": case_weight,
    })
overall_nrmse = pd.DataFrame(overall_rows).sort_values("equal_regime_weight_overall_nrmse")
display(overall_nrmse.round(3))

## Analysis stage 15

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

## Analysis stage 16

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
fnn_72 =table_72 [table_72 ["model"].eq ("FNN")].set_index ("regime")
fnn_abs_ratio =fnn_72 .loc ["Kuroshio Core","rmse_km"]/fnn_72 .loc ["Shelf","rmse_km"]
fnn_norm_ratio =fnn_72 .loc ["Kuroshio Core","nrmse"]/fnn_72 .loc ["Shelf","nrmse"]
fnn_boot =bootstrap_summary [bootstrap_summary ["model"].eq ("FNN")].iloc [0 ]

summary_text =f"""
### \u6838\u5fc3\u7ed3\u679c

- \u5728\u7edd\u5bf9\u5c3a\u5ea6\u4e0a，global FNN\u7684Kuroshio 72 h RMSE\u662fShelf\u7684 **{fnn_abs_ratio :.2f}\u500d**。
- \u4f7f\u7528\u5404regime training targets\u7684\u4e8c\u7ef4\u53d8\u5316\u5c3a\u5ea6\u6807\u51c6\u5316\u540e，Kuroshio/Shelf FNN NRMSE\u6bd4\u503c\u4e3a **{fnn_norm_ratio :.2f}**。
- \u56e0\u6b64\u6807\u51c6\u5316\u7ed3\u679c\u53ef\u4ee5\u533a\u5206“\u53ea\u662f\u79fb\u52a8\u5c3a\u5ea6\u66f4\u5927”\u4e0e“\u76f8\u5bf9\u9884\u6d4b\u80fd\u529b\u4e5f\u66f4\u5dee”。
- Kuroshio-only FNN\u76f8\u5bf9\u4e8eglobal FNN\u7684RMSE\u6539\u5584\u4e3a **{fnn_boot .rmse_improvement_pct :.2f}%**，
  drifter-cluster bootstrap 95%\u533a\u95f4\u4e3a **[{fnn_boot .improvement_ci_lower_95 :.2f}%, {fnn_boot .improvement_ci_upper_95 :.2f}%]**。
- \u662f\u5426\u53ef\u79f0\u4e3a\u7a33\u5b9a\u6539\u5584，\u5e94\u6839\u636e\u8be5\u533a\u95f4\u662f\u5426\u5b8c\u5168\u9ad8\u4e8e0，\u800c\u4e0d\u80fd\u53ea\u770b5.66%\u7684\u70b9\u4f30\u8ba1。

### \u63a8\u8350\u7684\u7ed3\u8bba\u7ed3\u6784

1. \u5148\u62a5\u544aabsolute km error，\u8bf4\u660e\u5b9e\u9645\u641c\u7d22\u8303\u56f4；
2. \u518d\u62a5\u544aNRMSE，\u63a7\u5236\u4e0d\u540c\u6d77\u6d0b\u73af\u5883\u7684\u8fd0\u52a8\u5c3a\u5ea6；
3. \u62a5\u544a\u76f8\u5bf9\u4e24\u4e2aphysical baselines\u7684forecast skill；
4. \u4f7f\u7528paired drifter-cluster bootstrap\u8bc4\u4ef7regional training\u6539\u5584\u662f\u5426\u7a33\u5b9a；
5. \u5c06\u5206\u9636\u6bb5FNN\u4fdd\u7559\u4e3atraining-CV\u63a7\u5236\u4e0b\u7684\u540e\u7eed\u5b9e\u9a8c，\u4e0d\u4f7f\u7528\u5f53\u524dtest\u66f2\u7ebf\u9009\u62e9\u9608\u503c。
"""
display (Markdown (summary_text ))

## Analysis stage 17

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.

In [ ]:
training_scale .to_csv (DATA_OUT /"training_regime_target_scale_by_lead.csv",index =False )
global_summary .to_csv (DATA_OUT /"global_models_absolute_normalised_skill_by_lead.csv",index =False )
table_72 .to_csv (DATA_OUT /"global_models_72h_absolute_normalised_skill.csv",index =False )
bootstrap_summary .to_csv (DATA_OUT /"kuroshio_regional_improvement_cluster_bootstrap.csv",index =False )
overall_nrmse .to_csv (DATA_OUT /"overall_nrmse_weighting_sensitivity.csv",index =False )
split_horizon_design .to_csv (DATA_OUT /"split_horizon_fnn_experiment_design.csv",index =False )

print ("\u4fdd\u5b58\u7684\u6570\u636e\u8868：")
for p in sorted (DATA_OUT .glob ("*")):
    print (" -",p .name )
print ("\n\u4fdd\u5b58\u7684\u56fe\u7247：")
for p in sorted (FIG_OUT .glob ("*")):
    print (" -",p .name )

## Analysis stage 18

This stage documents the next part of the analysis. The following code and retained outputs provide the exact computational record.